In [ ]:
def scan_dataset(raw_dir):
    """Each class folder here has the layout:
        <ClassName>/images/*.jpg   <- what we want
        <ClassName>/labels/*.txt   <- YOLO-format label files, ignored for classification
    Falls back to scanning the class folder directly if no images/ subfolder is found.
    """
    raw_dir = Path(raw_dir)
    class_dirs = sorted([d for d in raw_dir.iterdir() if d.is_dir()])
    data = {}
    for cd in class_dirs:
        img_dir = cd / "images" if (cd / "images").is_dir() else cd
        files = [p for p in img_dir.rglob("*") if p.suffix.lower() in IMG_EXTS]
        data[cd.name] = files
    return data

data_by_class = scan_dataset(RAW_DIR)
counts = {k: len(v) for k, v in data_by_class.items()}
total = sum(counts.values())

print(f"Classes: {len(counts)} | Total images: {total}\n")
for cls, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {cls:25s} {n:5d}  ({100*n/total:5.1f}%)")

imbalance_ratio = max(counts.values()) / max(1, min(counts.values()))
print(f"\nImbalance ratio (max/min class): {imbalance_ratio:.1f}x")
if imbalance_ratio > 5:
    print("=> Significant class imbalance. We mitigate this below with class-weighted loss.")


In [ ]:
# Class distribution bar chart
sorted_items = sorted(counts.items(), key=lambda x: -x[1])
plt.figure(figsize=(11, 6))
plt.bar([k for k, _ in sorted_items], [v for _, v in sorted_items], color="#2b6cb0")
plt.xticks(rotation=75, ha="right")
plt.ylabel("Number of images")
plt.title("Class distribution — Nepali Vehicle Dataset")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "class_distribution.png", dpi=150)
plt.show()


In [ ]:
# Sample grid — one image per class
classes_list = list(data_by_class.keys())
cols = 6
rows = (len(classes_list) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*2.2, rows*2.4))
axes = axes.flatten()
for i, cls in enumerate(classes_list):
    ax = axes[i]
    files = data_by_class[cls]
    if files:
        try:
            img = Image.open(files[0]).convert("RGB")
            ax.imshow(img)
        except Exception:
            pass
    ax.set_title(cls, fontsize=8)
    ax.axis("off")
for j in range(len(classes_list), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "sample_grid.png", dpi=150)
plt.show()


In [ ]:
# Image size distribution (sampled)
widths, heights = [], []
for files in data_by_class.values():
    for p in files[:15]:
        try:
            with Image.open(p) as im:
                widths.append(im.width); heights.append(im.height)
        except Exception:
            continue

print(f"Sampled {len(widths)} images.")
print(f"Width : min={min(widths)} max={max(widths)} mean={np.mean(widths):.0f}")
print(f"Height: min={min(heights)} max={max(heights)} mean={np.mean(heights):.0f}")

plt.figure(figsize=(5,5))
plt.scatter(widths, heights, alpha=0.4, s=10)
plt.xlabel("Width (px)"); plt.ylabel("Height (px)")
plt.title("Sampled image dimensions")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "image_size_scatter.png", dpi=150)
plt.show()
